# Lab 9 — News Summarization using Transformers
**Name:** R.K.Larika S.Harshini
**Roll Number:** CS23B1028 CS23B1050

## 1. Transformer Model Variants & Comparison

In [25]:
import pandas as pd

models_data = [
    {"Model": "BERT-base",          "Task": "Classification, QA, NER",         "Max Tokens": 512,   "Parameters": "110M"},
    {"Model": "BERT-large",         "Task": "Classification, QA, NER",         "Max Tokens": 512,   "Parameters": "340M"},
    {"Model": "GPT-2",              "Task": "Text Generation",                  "Max Tokens": 1024,  "Parameters": "117M"},
    {"Model": "GPT-3",              "Task": "Text Generation, Summarization",   "Max Tokens": 4096,  "Parameters": "175B"},
    {"Model": "T5-small",           "Task": "Summarization, Translation, QA",  "Max Tokens": 512,   "Parameters": "60M"},
    {"Model": "T5-base",            "Task": "Summarization, Translation, QA",  "Max Tokens": 512,   "Parameters": "220M"},
    {"Model": "T5-large",           "Task": "Summarization, Translation, QA",  "Max Tokens": 512,   "Parameters": "770M"},
    {"Model": "BART-base",          "Task": "Summarization, Translation",       "Max Tokens": 1024,  "Parameters": "139M"},
    {"Model": "BART-large",         "Task": "Summarization, Translation",       "Max Tokens": 1024,  "Parameters": "406M"},
    {"Model": "BART-large-cnn",     "Task": "News Summarization",               "Max Tokens": 1024,  "Parameters": "406M"},
    {"Model": "BART-large-xsum",    "Task": "Abstractive Summarization",        "Max Tokens": 1024,  "Parameters": "406M"},
    {"Model": "Pegasus-large",      "Task": "Abstractive Summarization",        "Max Tokens": 1024,  "Parameters": "568M"},
    {"Model": "DistilBART-cnn-6-6", "Task": "News Summarization",               "Max Tokens": 1024,  "Parameters": "230M"},
    {"Model": "RoBERTa-base",       "Task": "Classification, NER, QA",          "Max Tokens": 512,   "Parameters": "125M"},
    {"Model": "ELECTRA-base",       "Task": "Classification, NER",              "Max Tokens": 512,   "Parameters": "110M"},
    {"Model": "XLNet-base",         "Task": "Classification, QA",               "Max Tokens": 512,   "Parameters": "117M"},
    {"Model": "LED-base",           "Task": "Long-doc Summarization",           "Max Tokens": 16384, "Parameters": "162M"},
    {"Model": "BigBird-base",       "Task": "Long-doc QA, Summarization",       "Max Tokens": 4096,  "Parameters": "132M"},
]

df_models = pd.DataFrame(models_data)
df_models

,Model,Task,Max Tokens,Parameters
0,BERT-base,"Classification, QA, NER",512,110M
1,BERT-large,"Classification, QA, NER",512,340M
2,GPT-2,Text Generation,1024,117M
3,GPT-3,"Text Generation, Summarization",4096,175B
4,T5-small,"Summarization, Translation, QA",512,60M
5,T5-base,"Summarization, Translation, QA",512,220M
6,T5-large,"Summarization, Translation, QA",512,770M
7,BART-base,"Summarization, Translation",1024,139M
8,BART-large,"Summarization, Translation",1024,406M
9,BART-large-cnn,News Summarization,1024,406M


### Selected Model: `facebook/bart-base`
- 139M parameters — fits in free Colab/Kaggle GPU RAM
- 1024 max tokens — covers most news articles after filtering
- Pre-trained on denoising; fine-tunes well for abstractive summarization

## 2. Setup

In [38]:
!pip install -q datasets transformers[torch] rouge-score nltk accelerate sentencepiece evaluate bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [27]:
import os, nltk, torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from rouge_score import rouge_scorer
import evaluate

nltk.download('punkt', quiet=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

Device: cuda


## 3. Load & Filter Dataset

In [28]:
ds = load_dataset("ILSUM/ILSUM-1.0", "English")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 12565
    })
    test: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 4487
    })
    validation: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 898
    })
})


In [29]:
# Inspect token length distribution to decide filter threshold
from transformers import BartTokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

sample = ds["train"].select(range(500))
lengths = [len(tokenizer(x["Article"])["input_ids"]) for x in sample]
print(f"Median: {int(np.median(lengths))}  |  90th pct: {int(np.percentile(lengths, 90))}  |  Max: {max(lengths)}")

Median: 546  |  90th pct: 1967  |  Max: 5064


In [30]:
# Keep articles whose Article token length is <= 512
# This retains the majority while staying within model limits with minimal info loss
MAX_INPUT_TOKENS = 512

def length_filter(example):
    toks = tokenizer(example["Article"], truncation=False)["input_ids"]
    return len(toks) <= MAX_INPUT_TOKENS

filtered = ds.filter(length_filter, num_proc=2)
print(filtered)
print(f"Train size after filtering: {len(filtered['train'])}")

DatasetDict({
    train: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 5971
    })
    test: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 2087
    })
    validation: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 431
    })
})
Train size after filtering: 5971


In [31]:
# Confirm train > 1000
assert len(filtered["train"]) > 1000, "Training set too small after filtering"

# For quick experimentation on free compute, cap training at 3000 samples
TRAIN_LIMIT = 3000
train_ds  = filtered["train"].shuffle(seed=42).select(range(min(TRAIN_LIMIT, len(filtered["train"]))))
val_ds    = filtered["validation"]
test_ds   = filtered["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 3000 | Val: 431 | Test: 2087


## 4. Tokenize

In [32]:
MAX_TARGET_TOKENS = 128

def preprocess(batch):
    # Modern way to tokenize for seq2seq: use text_target for labels
    model_inputs = tokenizer(
        batch["Article"],
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=batch["Summary"],
        max_length=MAX_TARGET_TOKENS,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

cols_to_remove = ["id", "Article", "Heading", "Summary"]

tok_train = train_ds.map(preprocess, batched=True, remove_columns=cols_to_remove)
tok_val   = val_ds.map(preprocess,   batched=True, remove_columns=cols_to_remove)
tok_test  = test_ds.map(preprocess,  batched=True, remove_columns=cols_to_remove)

print(tok_train[0].keys())

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2087 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


## 5. Load Pre-trained BART Model

In [33]:
MODEL_NAME = "facebook/bart-base"
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.1f}M")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Parameters: 139.4M


## 6. Fine-tune

In [35]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./bart-ilsum-en",
    num_train_epochs            = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps                = 200,
    weight_decay               = 0.01,
    learning_rate              = 5e-5,
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model      = "eval_loss",
    predict_with_generate      = True,
    generation_max_length      = MAX_TARGET_TOKENS,
    fp16                       = torch.cuda.is_available(),
    logging_steps              = 50,
    report_to                  = "none",
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tok_train,
    eval_dataset    = tok_val,
    data_collator   = data_collator,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.392334,0.356881
2,0.357120,0.333678
3,0.308862,0.319403


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1125, training_loss=0.38806860054863823, metrics={'train_runtime': 336.5462, 'train_samples_per_second': 26.742, 'train_steps_per_second': 3.343, 'total_flos': 2501267567984640.0, 'train_loss': 0.38806860054863823, 'epoch': 3.0})

In [36]:
trainer.save_model("./bart-ilsum-en-final")
tokenizer.save_pretrained("./bart-ilsum-en-final")
print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


## 7. Evaluation Metrics

| Metric | What it measures |
|---|---|
| **ROUGE-1** | Unigram overlap (content coverage) |
| **ROUGE-2** | Bigram overlap (fluency/phrase match) |
| **ROUGE-L** | Longest common subsequence (structure) |
| **BERTScore** | Semantic similarity via contextual embeddings |
| **BLEU** | N-gram precision (more common in translation) |
| **METEOR** | Alignment + synonym matching |

We use **ROUGE-1, ROUGE-2, ROUGE-L** as primary metrics and **BERTScore** as a semantic check.

## 8. Evaluate on Test Set

In [39]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [42]:
# Generate summaries for test set in batches
model.eval()
BATCH_SIZE = 16
all_preds, all_refs = [], []

for i in range(0, len(test_ds), BATCH_SIZE):
    # Correctly slice the dataset and extract the 'Article' column as a list of strings
    batch = test_ds[i : i + BATCH_SIZE]
    articles = batch["Article"]

    enc = tokenizer(
        articles,
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_TOKENS,
            num_beams=4,
            early_stopping=True
        )

    preds = tokenizer.batch_decode(out, skip_special_tokens=True)
    all_preds.extend(preds)
    all_refs.extend(batch["Summary"])

print(f"Generated {len(all_preds)} summaries.")

Generated 2087 summaries.


In [43]:
# ROUGE
rouge_results = rouge.compute(predictions=all_preds, references=all_refs, use_stemmer=True)
for k, v in rouge_results.items():
    print(f"{k}: {v:.4f}")

rouge1: 0.4860
rouge2: 0.3616
rougeL: 0.4384
rougeLsum: 0.4382


In [44]:
# BERTScore (uses distilbert by default — fast)
bs_results = bertscore.compute(
    predictions=all_preds, references=all_refs, lang="en", model_type="distilbert-base-uncased"
)
print(f"BERTScore  Precision: {np.mean(bs_results['precision']):.4f}")
print(f"BERTScore  Recall:    {np.mean(bs_results['recall']):.4f}")
print(f"BERTScore  F1:        {np.mean(bs_results['f1']):.4f}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore  Precision: 0.8465
BERTScore  Recall:    0.8316
BERTScore  F1:        0.8385


In [45]:
# Qualitative check
for idx in range(3):
    print(f"--- Example {idx+1} ---")
    print("Article  :", test_ds[idx]["Article"][:300], "...")
    print("Reference:", test_ds[idx]["Summary"])
    print("Predicted:", all_preds[idx])
    print()

--- Example 1 ---
Article  : Indian-origin boy finds millions of years old fossil in UK gardenA six-year-old Indian-origin boy says he is “really excited” after he found a fossil from millions of years ago while digging in his garden in the West Midlands region of England. Siddak Singh Jhamat, known as Sid, was using a fossil-h ...
Reference: Siddak Singh Jhamat, known as Sid, was using a fossil-hunting set he received as a Christmas present when he came across a rock that looked like a horn.
Predicted: Siddak Singh Jhamat, known as Sid, was using a fossil-hunting set he received as a Christmas present when he came across a rock that looked like a horn.

--- Example 2 ---
Article  : Representative ImageA 38-year-old Indian man has been charged with conspiring to smuggle six of his countrymen into the US on commercial airline flights, authorities said.Bhavin Patel was arrested at the Newark Liberty International Airport last week and was charged on Monday with one count of consp ...
Ref